## Example 1: Basic gRPC span exporter

In [ ]:
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import InMemoryMetricReader
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, SpanExportResult
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

print("Example 1 - Basic gRPC span exporter")

# Nothing listens on 127.0.0.1:9 (the discard port), so every export below must return
# FAILURE instead of raising or hanging; timeout=1 keeps the exporters' retry loop short.
UNREACHABLE = "127.0.0.1:9"


def record_spans(*names):
    """Finish one span per name with a local tracer provider and return the finished spans."""
    store = InMemorySpanExporter()
    provider = TracerProvider()
    provider.add_span_processor(SimpleSpanProcessor(store))
    tracer = provider.get_tracer("example", "1.0")
    for name in names:
        with tracer.start_as_current_span(name) as span:
            span.set_attribute("example.name", name)
    return store.get_finished_spans()


def record_metrics(resource=None):
    """Record a counter with a local meter provider and return the collected MetricsData."""
    reader = InMemoryMetricReader()
    provider = MeterProvider(metric_readers=[reader], **({"resource": resource} if resource else {}))
    counter = provider.get_meter("example", version="1.0").create_counter("requests")
    counter.add(1, {"method": "GET"})
    return reader.get_metrics_data()


# Create OTLP gRPC span exporter
otlp_exporter = OTLPSpanExporter(endpoint=f"http://{UNREACHABLE}", insecure=True, timeout=1)

print(f"Created exporter: {type(otlp_exporter).__name__}")
print(f"Endpoint: http://{UNREACHABLE}")

spans = record_spans("test_span")
assert spans[0].attributes["example.name"] == "test_span"
result = otlp_exporter.export(spans)
print(f"export(): {result}")
assert result is SpanExportResult.FAILURE, result
otlp_exporter.shutdown()

print("Example 1 completed successfully")

## Example 2: gRPC metrics exporter

In [ ]:
# Reuses UNREACHABLE and record_metrics() from Example 1.
from opentelemetry.exporter.otlp.proto.grpc.metric_exporter import OTLPMetricExporter
from opentelemetry.sdk.metrics.export import MetricExportResult

print("Example 2 - gRPC metrics exporter")

# Create OTLP gRPC metrics exporter
metrics_exporter = OTLPMetricExporter(endpoint=f"http://{UNREACHABLE}", insecure=True, timeout=1)

print(f"Created exporter: {type(metrics_exporter).__name__}")

# Record a counter and export the collected metrics
result = metrics_exporter.export(record_metrics(), timeout_millis=1000)
print(f"export(): {result}")
assert result is MetricExportResult.FAILURE, result
metrics_exporter.shutdown()

print("Example 2 completed successfully")

## Example 3: Exporter with headers

In [ ]:
# Reuses UNREACHABLE and record_spans() from Example 1.
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace.export import SpanExportResult

print("Example 3 - Exporter with headers")

# Create exporter with custom headers (placeholder values)
exporter = OTLPSpanExporter(
    endpoint=f"http://{UNREACHABLE}",
    insecure=True,
    timeout=1,
    headers={"authorization": "Bearer token123", "x-custom-header": "value"}
)

print(f"Created exporter with custom headers")
print(f"Headers: authorization, x-custom-header")

result = exporter.export(record_spans("auth_span"))
print(f"export(): {result}")
assert result is SpanExportResult.FAILURE, result
exporter.shutdown()

print("Example 3 completed successfully")

## Example 4: Batch span processor

In [ ]:
# Reuses UNREACHABLE from Example 1.
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor, SpanExportResult

print("Example 4 - Batch span processor")

class RecordingExporter(OTLPSpanExporter):
    """The OTLP exporter, plus a record of every batch the processor hands it."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.batches = []

    def export(self, spans):
        result = super().export(spans)
        self.batches.append((len(spans), result))
        return result


exporter = RecordingExporter(endpoint=f"http://{UNREACHABLE}", insecure=True, timeout=1)

# Configure batch processor (small export batches so three spans need two exports)
processor = BatchSpanProcessor(
    exporter,
    max_queue_size=2048,
    max_export_batch_size=2
)
provider = TracerProvider()
provider.add_span_processor(processor)
tracer = provider.get_tracer("batch_example", "1.0")
for i in range(3):
    with tracer.start_as_current_span(f"batched_span_{i}"):
        pass

assert provider.force_flush(timeout_millis=10000), "force_flush timed out"
provider.shutdown()
print(f"Batches handed to the exporter (size, result): {exporter.batches}")
assert sum(size for size, _ in exporter.batches) == 3, exporter.batches
assert all(size <= 2 for size, _ in exporter.batches), exporter.batches
assert all(result is SpanExportResult.FAILURE for _, result in exporter.batches), exporter.batches

print("Example 4 completed successfully")

## Example 5: Multiple exporters

In [ ]:
# Reuses UNREACHABLE, record_spans() and record_metrics() from Example 1.
from opentelemetry.exporter.otlp.proto.grpc.metric_exporter import OTLPMetricExporter
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.metrics.export import MetricExportResult
from opentelemetry.sdk.trace.export import SpanExportResult

print("Example 5 - Multiple exporters")

# Create trace exporter
trace_exporter = OTLPSpanExporter(endpoint=f"http://{UNREACHABLE}", insecure=True, timeout=1)

# Create metrics exporter
metrics_exporter = OTLPMetricExporter(endpoint=f"http://{UNREACHABLE}", insecure=True, timeout=1)

print(f"Trace exporter: {type(trace_exporter).__name__}")
print(f"Metrics exporter: {type(metrics_exporter).__name__}")

trace_result = trace_exporter.export(record_spans("multi_span"))
metrics_result = metrics_exporter.export(record_metrics(), timeout_millis=1000)
print(f"Trace export: {trace_result}; metrics export: {metrics_result}")
assert trace_result is SpanExportResult.FAILURE, trace_result
assert metrics_result is MetricExportResult.FAILURE, metrics_result
trace_exporter.shutdown()
metrics_exporter.shutdown()

print("Example 5 completed successfully")